In [2]:
# Cell 1: Imports and Setup
import sys
import os
import random
from collections import defaultdict
from typing import List

import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import numpy as np

# Add your src directory if needed
sys.path.append(os.path.abspath("../src"))

from abstractions3d.dsl.core import Shape3D
from abstractions3d.dsl.instantiation import instantiate_3d
from abstractions3d.dsl.nodes import Rect3D, Move3D, SymTrans3D, SymRef3D, Union3D
from abstractions3d.primitives.visualization import visualize_boxes_3d
from abstractions3d.data.blueprint import Table3D, Chair3D, Bench3D

random.seed(42)
torch.manual_seed(42)

In [3]:
# Cell 2: Dataset Generation
def sample_uniform(low: float, high: float) -> float:
    return random.uniform(low, high)

def generate_dataset(num_samples: int = 50) -> List[Union3D]:
    dataset = []
    for _ in range(num_samples):
        dataset.append(Table3D(
            top_length=sample_uniform(3.0, 6.0),
            top_depth=sample_uniform(1.5, 3.0),
            top_thickness=sample_uniform(0.1, 0.3),
            leg_length=sample_uniform(0.2, 0.4),
            leg_depth=sample_uniform(0.2, 0.4),
            leg_height=sample_uniform(1.0, 2.0)
        ))
        dataset.append(Chair3D(
            seat_length=sample_uniform(1.0, 2.0),
            seat_depth=sample_uniform(1.0, 2.0),
            seat_thickness=sample_uniform(0.2, 0.5),
            leg_length=sample_uniform(0.1, 0.3),
            leg_depth=sample_uniform(0.1, 0.3),
            leg_height=sample_uniform(0.5, 1.0),
            backrest_height=sample_uniform(1.0, 2.0),
            backrest_thickness=sample_uniform(0.1, 0.3)
        ))
        backrest_height = sample_uniform(0.0, 1.5)
        dataset.append(Bench3D(
            seat_length=sample_uniform(2.0, 5.0),
            seat_depth=sample_uniform(0.5, 1.0),
            seat_thickness=sample_uniform(0.2, 0.5),
            leg_length=sample_uniform(0.1, 0.3),
            leg_depth=sample_uniform(0.1, 0.3),
            leg_height=sample_uniform(0.5, 1.0),
            backrest_height=backrest_height,
            backrest_thickness=sample_uniform(0.0, 0.3) if backrest_height > 0 else 0.0
        ))
    return dataset

dataset = generate_dataset(50)
print(f"Generated {len(dataset)} shapes.")

Generated 150 shapes.


In [4]:
# Cell 3: Flatten Shapes to Tensors
def shape_to_tensor(shape: Shape3D) -> torch.Tensor:
    """Convert a Shape3D to a tensor of box centers and scales (N,6)."""
    boxes = shape.get_box3d_list()
    tensor_list = [torch.cat([box.center, box.scale]) for box in boxes]
    return torch.cat(tensor_list) if tensor_list else torch.zeros(6)

# Find max length of flattened tensors for padding
max_len = max(shape_to_tensor(s).numel() for s in dataset)
print(f"Max flattened length: {max_len}")

def pad_tensor(tensor: torch.Tensor, target_len: int) -> torch.Tensor:
    """Zero-pad tensor to target length."""
    padded = torch.zeros(target_len)
    padded[:tensor.numel()] = tensor
    return padded

shape_tensors = torch.stack([pad_tensor(shape_to_tensor(s), max_len) for s in dataset])
print(f"Shape tensor dataset: {shape_tensors.shape}")

# Cell 4: Simple Autoencoder
class ShapeAutoencoder(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int = 32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        recon = self.decoder(z)
        return recon, z

# Cell 5: Training Pipeline
input_dim = shape_tensors.shape[1]
latent_dim = 32
model = ShapeAutoencoder(input_dim, latent_dim)
optimizer = AdamW(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
batch_size = 16
epochs = 50

dataloader = DataLoader(TensorDataset(shape_tensors), batch_size=batch_size, shuffle=True)

for epoch in range(epochs):
    total_loss = 0.0
    for batch, in dataloader:
        optimizer.zero_grad()
        recon, _ = model(batch)
        loss = criterion(recon, batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(dataloader):.6f}")

# Cell 6: Encode / Decode Example
example_shape = dataset[0]
example_tensor = pad_tensor(shape_to_tensor(example_shape), max_len).unsqueeze(0)
with torch.no_grad():
    latent = model.encoder(example_tensor)
    reconstructed = model.decoder(latent)
print("Original tensor:", example_tensor)
print("Reconstructed tensor:", reconstructed)

Max flattened length: 36
Shape tensor dataset: torch.Size([150, 36])
Epoch 1/50 - Loss: 1.036982
Epoch 2/50 - Loss: 0.541523
Epoch 3/50 - Loss: 0.262701
Epoch 4/50 - Loss: 0.194261
Epoch 5/50 - Loss: 0.169397
Epoch 6/50 - Loss: 0.147841
Epoch 7/50 - Loss: 0.117851
Epoch 8/50 - Loss: 0.081895
Epoch 9/50 - Loss: 0.054516
Epoch 10/50 - Loss: 0.040726
Epoch 11/50 - Loss: 0.029086
Epoch 12/50 - Loss: 0.019086
Epoch 13/50 - Loss: 0.015361
Epoch 14/50 - Loss: 0.013515
Epoch 15/50 - Loss: 0.012868
Epoch 16/50 - Loss: 0.012353
Epoch 17/50 - Loss: 0.011718
Epoch 18/50 - Loss: 0.010743
Epoch 19/50 - Loss: 0.010414
Epoch 20/50 - Loss: 0.010259
Epoch 21/50 - Loss: 0.009267
Epoch 22/50 - Loss: 0.008505
Epoch 23/50 - Loss: 0.008117
Epoch 24/50 - Loss: 0.007340
Epoch 25/50 - Loss: 0.006474
Epoch 26/50 - Loss: 0.005652
Epoch 27/50 - Loss: 0.005505
Epoch 28/50 - Loss: 0.004888
Epoch 29/50 - Loss: 0.004372
Epoch 30/50 - Loss: 0.004139
Epoch 31/50 - Loss: 0.003919
Epoch 32/50 - Loss: 0.003930
Epoch 33/50 

In [5]:
# Recursive Abstraction class for 3D shapes
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset
from typing import List, Tuple, Any

class NodeEncoder(nn.Module):
    """Encodes a single node's parameters + child latents."""
    def __init__(self, param_dim: int, child_dim: int, latent_dim: int = 32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(param_dim + child_dim, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, param_dim + child_dim)
        )

    def forward(self, params: torch.Tensor, child_latent: torch.Tensor):
        x = torch.cat([params, child_latent], dim=-1)
        z = self.encoder(x)
        recon = self.decoder(z)
        return recon, z

class RecursiveAbstraction:
    def __init__(self, latent_dim: int = 32, lr: float = 1e-3):
        self.latent_dim = latent_dim
        self.lr = lr
        self.node_encoders = {}  # dict[node_type] -> NodeEncoder
        self.optimizers = {}

    def get_params_tensor(self, node) -> torch.Tensor:
        """Convert node parameters to tensor (float + categorical encoding)."""
        if hasattr(node, "param_tuple"):
            _, params = node.param_tuple()
            flat = []
            for p in params:
                if isinstance(p, (int, float)):
                    flat.append(float(p))
                elif isinstance(p, str):
                    flat.append({"x":0.0, "y":1.0, "z":2.0}.get(p, -1.0))
                elif isinstance(p, Shape3D):
                    flat.append(0.0)  # placeholder for child
                else:
                    flat.append(0.0)
            return torch.tensor(flat, dtype=torch.float32)
        return torch.zeros(1)

    def encode_node(self, node) -> torch.Tensor:
        """Recursively encode node + children."""
        child_latents = []
        for child in getattr(node, "children", []):
            child_latents.append(self.encode_node(child))
        child_latent = torch.cat(child_latents) if child_latents else torch.zeros(0)

        params = self.get_params_tensor(node)
        param_dim = params.numel()
        child_dim = child_latent.numel()

        node_type = type(node).__name__
        if node_type not in self.node_encoders:
            self.node_encoders[node_type] = NodeEncoder(param_dim, child_dim, self.latent_dim)
            self.optimizers[node_type] = AdamW(self.node_encoders[node_type].parameters(), lr=self.lr)

        encoder = self.node_encoders[node_type]
        recon, z = encoder(params.unsqueeze(0), child_latent.unsqueeze(0))
        return z.squeeze(0)

    def train(self, dataset: List[Shape3D], epochs: int = 50):
        criterion = nn.MSELoss()
        for epoch in range(epochs):
            total_loss = 0.0
            for node in dataset:
                # Encode recursively and compute reconstruction loss per node type
                def train_node(node):
                    child_latents = []
                    for child in getattr(node, "children", []):
                        child_latents.append(train_node(child))
                    child_latent = torch.cat(child_latents) if child_latents else torch.zeros(0)
                    params = self.get_params_tensor(node)
                    param_dim = params.numel()
                    child_dim = child_latent.numel()

                    node_type = type(node).__name__
                    encoder = self.node_encoders[node_type]
                    optimizer = self.optimizers[node_type]

                    optimizer.zero_grad()
                    recon, z = encoder(params.unsqueeze(0), child_latent.unsqueeze(0))
                    target = torch.cat([params, child_latent]).unsqueeze(0)
                    loss = criterion(recon, target)
                    loss.backward()
                    optimizer.step()
                    return z.squeeze(0)

                total_loss += train_node(node).sum().item()
            print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(dataset):.6f}")

    def encode_dataset(self, dataset: List[Shape3D]) -> List[torch.Tensor]:
        """Return latent vectors for all shapes in dataset."""
        latents = []
        for node in dataset:
            z = self.encode_node(node)
            latents.append(z)
        return latents

In [ ]:
from collections import defaultdict
from typing import Union, List

ShapeType = Union['Shape3D', List['Shape3D']]

def add(structures1: defaultdict, structures2: defaultdict) -> defaultdict:
    """
    Merges two defaultdict(list) structures by appending values for matching keys.
    """
    for key, value in structures2.items():
        structures1[key] += value
    return structures1


def get_singletons(shapes: ShapeType) -> defaultdict:
    """
    Recursively extracts all singleton (non-composite) shapes.
    """
    singletons = defaultdict(list)

    if isinstance(shapes, list):
        for shape in shapes:
            singletons = add(singletons, get_singletons(shape))
        return singletons

    shape_type, parameters = shapes.param_tuple()
    singletons[shape_type.__name__].append(parameters)

    # safely handle children
    children = getattr(shapes, 'children', [])
    if children is None:
        children = []

    if isinstance(shapes, (Move3D, SymTrans3D, SymRef3D)) and children:
        singletons = add(singletons, get_singletons(children[0]))
    elif isinstance(shapes, Union3D):
        for child in children:
            singletons = add(singletons, get_singletons(child))

    return singletons


def get_pairs(shapes: ShapeType) -> defaultdict:
    """
    Recursively extracts all binary composite structures.
    """
    pairs = defaultdict(list)

    if isinstance(shapes, list):
        for shape in shapes:
            pairs = add(pairs, get_pairs(shape))
        return pairs

    children = getattr(shapes, 'children', [])
    if children is None:
        children = []

    if isinstance(shapes, (Move3D, SymTrans3D, SymRef3D)) and children:
        return get_pairs(children[0])
    elif isinstance(shapes, Union3D):
        if len(children) != 2:
            return defaultdict(list)
        type_, (child1, child2) = shapes.param_tuple()
        type1, params1 = child1.param_tuple()
        type2, params2 = child2.param_tuple()
        type_str = f"{type_.__name__}({type1.__name__}, {type2.__name__})"
        current_structures = defaultdict(list)
        current_structures[type_str].append(params1 + params2)
        # include recursively all child pairs
        current_structures = add(current_structures, get_pairs(children[0]))
        current_structures = add(current_structures, get_pairs(children[1]))
        return current_structures

    return defaultdict(list)

In [7]:
# Cell 3: Extract Singleton and Pair Structures
singletons = get_singletons(dataset)
pairs = get_pairs(dataset)
structures = add(singletons, pairs)

print(f"Singletons: {list(singletons.keys())}")
print(f"Pairs: {list(pairs.keys())}")

Singletons: ['Union3D', 'Move3D', 'Rect3D', 'SymRef3D', 'Union3D(Move3D, SymRef3D)', 'Union3D(Move3D, Union3D)', 'Union3D(SymRef3D, Move3D)']
Pairs: ['Union3D(Move3D, SymRef3D)', 'Union3D(Move3D, Union3D)', 'Union3D(SymRef3D, Move3D)']


In [8]:
structures

defaultdict(list,
            {'Union3D': [(<abstractions3d.dsl.nodes.Move3D at 0x1517c6750>,
               <abstractions3d.dsl.nodes.SymRef3D at 0x1517c6810>),
              (<abstractions3d.dsl.nodes.Move3D at 0x1517c68a0>,
               <abstractions3d.dsl.nodes.Union3D at 0x1517c69f0>),
              (<abstractions3d.dsl.nodes.SymRef3D at 0x1517c6960>,
               <abstractions3d.dsl.nodes.Move3D at 0x1517c69c0>),
              (<abstractions3d.dsl.nodes.Move3D at 0x1517c6a80>,
               <abstractions3d.dsl.nodes.Union3D at 0x1517c6bd0>),
              (<abstractions3d.dsl.nodes.SymRef3D at 0x1517c6b40>,
               <abstractions3d.dsl.nodes.Move3D at 0x1517c6ba0>),
              (<abstractions3d.dsl.nodes.Move3D at 0x1517c6c60>,
               <abstractions3d.dsl.nodes.SymRef3D at 0x1517c6d20>),
              (<abstractions3d.dsl.nodes.Move3D at 0x1517c6db0>,
               <abstractions3d.dsl.nodes.Union3D at 0x1517c6f00>),
              (<abstractions3d.dsl.nodes.Sy